# 002_enrich

**Author:** Wayne Kirk Schmidt  
**Email:** wayne.kirk.schmidt@gmail.com

## Purpose
-------
002_enrich.ipynb is the second stage of the cryptocurrency statistical arbitrage research pipeline.

This stage transforms the raw event panels produced by 001_download into analysis-ready data.
The notebook computes reaction metrics that describe how other assets respond after a price
movement in a reference coin.

No filtering or strategy assumptions are applied in this stage. All transformations remain
reversible so later notebooks can test multiple hypotheses.

## Inputs
------
output/001_download/<coin>.event_panel.pkl

Each file contains rows structured as:

ref_coin | date | ref_close | target_coin | target_close_lags

Where target_close_lags is a list of future closes:

[ close(t+1), close(t+2), ..., close(t+N) ]


## Responsibilities
----------------
- load event panel datasets for each reference coin
- convert future closes into return series
- compute reaction statistics for each event
- construct a unified analysis dataset across all assets


## Outputs
-------
output/002_enrich/

Example artifacts:
reaction_panel.pkl
reaction_metrics.pkl


## Notes
-----
This stage intentionally preserves all events.

Filtering based on magnitude, quartiles, or shock definitions is deferred
to later analysis so that multiple hypotheses can be tested without
regenerating the event panels.

### 1. Imports and Environment Setup
### Provide the necessary imports required for to to proceed.   

In [1]:
import pandas as pd
import numpy as np
import pickle
from datetime import datetime
import math
from pathlib import Path

### 2. Prepare the environment for the notebook

In [2]:
startdate = "2023-01-01"
trading_days = 252
frequency = "1d"

universe = [
    "BTCUSDT",   # Bitcoin
    "ETHUSDT",   # Ethereum
    "BNBUSDT",   # Binance Coin
    "SOLUSDT",   # Solana
    "XRPUSDT",   # Ripple
    "ADAUSDT",   # Cardano
    "DOGEUSDT",  # Dogecoin
    "AVAXUSDT",  # Avalanche
    "LTCUSDT"    # Litecoin
]

execution_delay = [0, 1, 2, 3]
execution_cost_bps = [20, 30, 40]

stage_label = "002_enrich"

OUTPUT_ROOT = Path("../output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_FILE = OUTPUT_ROOT / "manifest.pkl"

DOWNLOAD_DIR = OUTPUT_ROOT / "001_download"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

ENRICH_DIR = OUTPUT_ROOT / "002_enrich"
ENRICH_DIR.mkdir(parents=True, exist_ok=True)

inspection_window = 20

observation_window_length = 10
observation_window = range(1, observation_window_length + 1)

holding_period = 1


### 2.1 Loading the manifest pickle file

In [3]:
MANIFEST_FILE = OUTPUT_ROOT / "manifest.pkl"

if MANIFEST_FILE.exists():
    manifest = pd.read_pickle(MANIFEST_FILE)
else:
    manifest = {}

manifest.setdefault(stage_label, {})

{}

### 3. Load the pickle files from the previous stage
Load all of the coin based pickle files, combine, and then output the combined results.

In [4]:
files = sorted(DOWNLOAD_DIR.glob("*.event_panel.pkl"))
print("files found:", len(files))
panels = []
for path in files:
    df = pd.read_pickle(path)
    panels.append(df)
events = pd.concat(panels, ignore_index=True)
print("rows:", len(events))
print("columns:", list(events.columns))
# persist raw event dataset
EVENTS_RAW_FILE = ENRICH_DIR / "events_raw.pkl"
events.to_pickle(EVENTS_RAW_FILE)
print("saved:", EVENTS_RAW_FILE)

files found: 9
rows: 69984
columns: ['ref_coin', 'date', 'ref_close', 'target_coin', 'target_close_lags']
saved: ../output/002_enrich/events_raw.pkl


### 4. Load the PRICES pickle file from the previous stage

In [5]:
PRICE_FILE = DOWNLOAD_DIR / "PRICES.pkl"

prices = pd.read_pickle(PRICE_FILE)

print("rows:", len(prices))
print("columns:", list(prices.columns))

coins = sorted(prices["coin"].unique())

print("number of coins:", len(coins))
print("coins:", coins)

prices.head()


# persist raw prices dataset
PRICES_RAW_FILE = ENRICH_DIR / "prices_raw.pkl"
prices.to_pickle(PRICES_RAW_FILE)

print("saved:", PRICES_RAW_FILE)

rows: 10390
columns: ['date', 'coin', 'open', 'close', 'volume']
number of coins: 9
coins: ['ADAUSDT', 'AVAXUSDT', 'BNBUSDT', 'BTCUSDT', 'DOGEUSDT', 'ETHUSDT', 'LTCUSDT', 'SOLUSDT', 'XRPUSDT']
saved: ../output/002_enrich/prices_raw.pkl


In [6]:
print(events.columns)
events.head()

Index(['ref_coin', 'date', 'ref_close', 'target_coin', 'target_close_lags'], dtype='object')


,ref_coin,date,ref_close,target_coin,target_close_lags
0,ADAUSDT,2023-07-14 00:00:00+00:00,0.3282,BTCUSDT,"[30288.46, 30225.15, 30154.58, 29864.32, 29925..."
1,ADAUSDT,2023-07-14 00:00:00+00:00,0.3282,ETHUSDT,"[1931.4, 1920.96, 1912.29, 1896.7, 1889.11, 18..."
2,ADAUSDT,2023-07-14 00:00:00+00:00,0.3282,BNBUSDT,"[249.1, 241.9, 243.7, 239.1, 240.3, 241.5, 241..."
3,ADAUSDT,2023-07-14 00:00:00+00:00,0.3282,SOLUSDT,"[27.36, 27.39, 26.85, 25.57, 26.33, 25.33, 25...."
4,ADAUSDT,2023-07-14 00:00:00+00:00,0.3282,XRPUSDT,"[0.7157, 0.7488, 0.739, 0.7825, 0.8211, 0.7948..."


### 5. Collect a vector of the items X days after the event for a given coin for each of the other coins

In [7]:
# enforce synchronized lag vectors across all coins
# compute vector length
events["vector_length"] = events["target_close_lags"].apply(len)

print("vector length distribution:")
print(events["vector_length"].value_counts())

# find dates where every coin has a full observation window
valid_dates = (
    events.groupby("date")["vector_length"]
    .min()
)

valid_dates = valid_dates[valid_dates == observation_window_length].index

# keep only synchronized dates
events = events[events["date"].isin(valid_dates)].reset_index(drop=True)

# remove helper column
events = events.drop(columns=["vector_length"])

print("rows after synchronized vector validation:", len(events))
events.head()

# persist synchronized events dataset
EVENTS_FILE = ENRICH_DIR / "events.pkl"
events.to_pickle(EVENTS_FILE)
print("saved:", EVENTS_FILE)

# register artifact in manifest
manifest[stage_label]["events"] = str(EVENTS_FILE)

vector length distribution:
vector_length
10    69984
Name: count, dtype: int64
rows after synchronized vector validation: 69984
saved: ../output/002_enrich/events.pkl


### 6. Now pivot the price table

Convert the long price dataset into a wide matrix where:

rows = dates  
columns = coins  
values = closing prices

This structure is required for vectorized statistical analysis across assets.

In [8]:
price_wide = prices.pivot(index="date", columns="coin", values="close")
price_wide.head()

coin,ADAUSDT,AVAXUSDT,BNBUSDT,BTCUSDT,DOGEUSDT,ETHUSDT,LTCUSDT,SOLUSDT,XRPUSDT
date,,,,,,,,,
2023-01-01 00:00:00+00:00,0.24989,10.87,244.1924,16617.57,0.070069,1199.97,70.69,9.9994,NaN
2023-01-02 00:00:00+00:00,0.25336,11.15,245.2174,16677.87,0.071352,1214.05,74.80,11.2914,NaN
2023-01-03 00:00:00+00:00,0.25297,11.37,246.0136,16674.12,0.070514,1214.68,75.52,13.3944,NaN
2023-01-04 00:00:00+00:00,0.26796,12.07,259.2094,16849.97,0.073260,1256.89,75.46,13.4225,NaN
2023-01-05 00:00:00+00:00,0.26891,11.75,256.5466,16832.48,0.071411,1251.03,74.06,13.4253,NaN


### 7. Compute daily returns

Convert the synchronized price matrix into a return matrix.

Daily returns are calculated using the classical percentage change formula:

r_t = (P_t / P_(t-1)) - 1

This produces a matrix of daily returns for each asset that will be used for volatility estimation and event detection.

In [9]:
returns = price_wide.pct_change()

print("returns shape:", returns.shape)
returns.head()

returns shape: (1176, 9)


coin,ADAUSDT,AVAXUSDT,BNBUSDT,BTCUSDT,DOGEUSDT,ETHUSDT,LTCUSDT,SOLUSDT,XRPUSDT
date,,,,,,,,,
2023-01-01 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-02 00:00:00+00:00,0.013886,0.025759,0.004198,0.003629,0.018313,0.011734,0.058141,0.129208,NaN
2023-01-03 00:00:00+00:00,-0.001539,0.019731,0.003247,-0.000225,-0.011753,0.000519,0.009626,0.186248,NaN
2023-01-04 00:00:00+00:00,0.059256,0.061566,0.053638,0.010546,0.038938,0.034750,-0.000794,0.002098,NaN
2023-01-05 00:00:00+00:00,0.003545,-0.026512,-0.010273,-0.001038,-0.025229,-0.004662,-0.018553,0.000209,NaN


### Section 8 — Identify First Valid Observations
Before synchronizing the dataset, we determine the first date where each asset has valid price and return observations.

Different assets begin trading at different times in the dataset. For example, some assets such as XRP may appear later than others. 
In addition, return calculations require a previous price observation, so valid return values naturally begin one period after the first price observation.

Identifying these start points allows us to determine where the dataset becomes fully populated across all assets. This information is used in the next step to synchronize the return matrix so that cross-asset comparisons are performed over a consistent time window.

In [10]:
PRICE_WIDE_FILE = ENRICH_DIR / "price_wide.pkl"
price_wide.to_pickle(PRICE_WIDE_FILE)
manifest[stage_label]["price_wide"] = str(PRICE_WIDE_FILE)
print("saved:", PRICE_WIDE_FILE)
print("shape:", price_wide.shape)

saved: ../output/002_enrich/price_wide.pkl
shape: (1176, 9)


### 9. Synchronize the return matrix

Assets in the dataset do not all begin trading on the same date.  

To ensure valid cross-asset comparisons, we remove rows that contain
missing values so that every remaining row contains returns for all coins.

This produces a full matrix where:

rows = trading dates  
columns = assets  
values = daily returns

The resulting dataset can now be used safely for volatility estimation and cross-asset event analysis.

In [11]:
returns_full = returns.dropna(how="any").copy()
print("returns_full shape:", returns_full.shape)
print("first synchronized date:", returns_full.index.min())
print("last synchronized date:", returns_full.index.max())
returns_full.head()
RETURNS_FULL_FILE = ENRICH_DIR / "returns_full.pkl"
returns_full.to_pickle(RETURNS_FULL_FILE)
manifest[stage_label]["returns_full"] = str(RETURNS_FULL_FILE)
print("saved:", RETURNS_FULL_FILE)

returns_full shape: (981, 9)
first synchronized date: 2023-07-15 00:00:00+00:00
last synchronized date: 2026-03-21 00:00:00+00:00
saved: ../output/002_enrich/returns_full.pkl


### 10. Estimate rolling volatility

To identify statistically significant price movements, we estimate the
recent volatility of each asset using a rolling standard deviation of returns.

Volatility is calculated using a 30-day rolling window so that each estimate
reflects the recent historical variability of returns.

The rolling window needs to have enough valid observations, so we will see the first 
29 rows will contain NaN values because the volatility estimate cannot be computed until enough past data exists.

In [12]:
rolling_sigma = returns_full.rolling(window=inspection_window).std()
rolling_sigma = rolling_sigma.dropna(how="any")
print("sigma matrix shape:", rolling_sigma.shape)
rolling_sigma.head()
SIGMA_FILE = ENRICH_DIR / "rolling_sigma.pkl"
rolling_sigma.to_pickle(SIGMA_FILE)
manifest[stage_label]["rolling_sigma"] = str(SIGMA_FILE)
print("saved:", SIGMA_FILE)

sigma matrix shape: (962, 9)
saved: ../output/002_enrich/rolling_sigma.pkl


### 11. Compute standardized returns (z-scores)

To identify unusually large price movements, returns are normalized by
their rolling volatility estimate.

This produces a standardized return (z-score) that measures how large
a movement is relative to the asset's recent volatility.

Large absolute z-scores indicate statistically unusual movements that
may propagate across assets.

In [13]:
aligned_returns = returns_full.loc[rolling_sigma.index]
z_scores = aligned_returns / rolling_sigma
print("z-score matrix shape:", z_scores.shape)
z_scores.head()
ZSCORE_FILE = ENRICH_DIR / "z_scores.pkl"
z_scores.to_pickle(ZSCORE_FILE)
manifest[stage_label]["z_scores"] = str(ZSCORE_FILE)
print("saved:", ZSCORE_FILE)

z-score matrix shape: (962, 9)
saved: ../output/002_enrich/z_scores.pkl


### 999. Persist the information to the pickle file

In [14]:
pd.to_pickle(manifest, MANIFEST_FILE)
print("manifest saved:", MANIFEST_FILE)
print("stages:", list(manifest.keys()))
print("artifacts in stage:", list(manifest[stage_label].keys()))

manifest saved: ../output/manifest.pkl
stages: ['001_download', '002_enrich']
artifacts in stage: ['events', 'price_wide', 'returns_full', 'rolling_sigma', 'z_scores']
